In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.linalg import expm

# ---------- Paths ----------
def find_base_dir():
    env = os.environ.get("PROJECT_DIR")
    if env and (Path(env) / "data" / "raw").exists():
        return Path(env)
    start = Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "data" / "raw").exists():
            return p
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        chosen = filedialog.askdirectory(title="Select the project folder (the one containing data/raw)")
        root.destroy()
    except Exception:
        chosen = input("Paste the project folder path: ").strip().strip('"')
    if chosen and (Path(chosen) / "data" / "raw").exists():
        return Path(chosen)
    raise FileNotFoundError("data/raw not found in the selected folder.")

BASE_DIR = find_base_dir()
RAW_DIR = BASE_DIR / "data" / "raw"
OUT_DIR = BASE_DIR / "outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("BASE_DIR:", BASE_DIR)

def safe_save(save_fn, filename):
    try:
        save_fn(OUT_DIR / filename)
        print("Saved:", OUT_DIR / filename)
    except OSError as e:
        fallback = Path.home() / "part3c_outputs"
        fallback.mkdir(exist_ok=True)
        save_fn(fallback / filename)
        print("Could not write to", OUT_DIR, "-", e)
        print("Saved to", fallback / filename, "instead.")

# ---------- Data ----------
ins = pd.read_csv(RAW_DIR / "insured_transitions.csv")

# Pool once per (district_id, origin) per README.txt
unique_origin = ins.drop_duplicates(subset=["district_id", "origin"])[
    ["district_id", "origin", "exposure_py", "initial_lives"]
]
pooled_exp = unique_origin.groupby("origin")["exposure_py"].sum().to_dict()
pooled_init = unique_origin.groupby("origin")["initial_lives"].sum().to_dict()
pooled_jumps = ins.groupby(["origin", "destination"])["transition_count"].sum().to_dict()

lam_HI = pooled_jumps[("H", "I")] / pooled_exp["H"]
lam_HD = pooled_jumps[("H", "D")] / pooled_exp["H"]
lam_IH = pooled_jumps[("I", "H")] / pooled_exp["I"]
lam_ID = pooled_jumps[("I", "D")] / pooled_exp["I"]

Q = np.array([
    [-(lam_HI + lam_HD),  lam_HI,            lam_HD],
    [ lam_IH,            -(lam_IH + lam_ID),  lam_ID],
    [ 0.0,                0.0,               0.0]
], dtype=float)

# 1. Compute P(1) and P(2) via matrix exponential
P1 = expm(Q * 1.0)
P2 = expm(Q * 2.0)

states = ["H", "I", "D"]
P1_df = pd.DataFrame(P1, index=states, columns=states)
P2_df = pd.DataFrame(P2, index=states, columns=states)

# 2. Numerical checks
assert (P1 >= 0.0).all() and (P2 >= 0.0).all(), "Negative probability encountered"
assert np.allclose(P1.sum(axis=1), 1.0, atol=1e-12), "P(1) rows do not sum to 1"
assert np.allclose(P2.sum(axis=1), 1.0, atol=1e-12), "P(2) rows do not sum to 1"
assert np.allclose(P1[2, :], [0.0, 0.0, 1.0], atol=1e-12), "D is not absorbing in P(1)"
assert np.allclose(P2[2, :], [0.0, 0.0, 1.0], atol=1e-12), "D is not absorbing in P(2)"

max_ck_diff = float(np.max(np.abs(P2 - (P1 @ P1))))
assert np.allclose(P2, P1 @ P1, atol=1e-12), f"P(2) != P(1)^2: max diff = {max_ck_diff}"

# 3. Expected occupancy from v0 = [N_H(0), N_I(0), N_D(0)]
v0 = np.array([pooled_init.get("H", 0.0), pooled_init.get("I", 0.0), pooled_init.get("D", 0.0)], dtype=float)
occ_0 = v0
occ_1 = v0 @ P1
occ_2 = v0 @ P2

occ_df = pd.DataFrame([
    {"year": 0, "H": occ_0[0], "I": occ_0[1], "D": occ_0[2], "total": occ_0.sum()},
    {"year": 1, "H": occ_1[0], "I": occ_1[1], "D": occ_1[2], "total": occ_1.sum()},
    {"year": 2, "H": occ_2[0], "I": occ_2[1], "D": occ_2[2], "total": occ_2.sum()},
])

# Show tables first, then save
print("\nP(1):")
display(P1_df)
print("P(2):")
display(P2_df)
print(f"Max |P(2) - P(1)^2| = {max_ck_diff:.3e}")
print("\nExpected occupancy:")
display(occ_df)

safe_save(lambda p: P1_df.to_csv(p), "part3c_P1.csv")
safe_save(lambda p: P2_df.to_csv(p), "part3c_P2.csv")
safe_save(lambda p: occ_df.to_csv(p, index=False), "part3c_expected_occupancy.csv")